In [2]:
def evaluate_ecc_security(p, a, b, curve_name="未命名"):
    """
    评估椭圆曲线 E(F_p): y^2 = x^3 + ax + b 的有效安全比特数
    返回 final_security (float)
    """
    print(f"\n--- 正在评估曲线：{curve_name} ---")
    print(f"p = {p}\na = {a}\nb = {b}")
    
    # 1. 检查奇异曲线攻击
    discriminant = (4 * a**3 + 27 * b**2) % p
    if discriminant == 0:
        print("[!] 警告: 判别式为 0，这是一条奇异曲线！")
        print("-> 威胁: ECDLP 将退化为有限域上的多项式时间问题。")
        return 0

    # 构造曲线并计算阶
    F = GF(p)
    try:
        E = EllipticCurve(F, [a, b])
        N = E.order()
    except Exception as e:
        print("[!] 曲线构造失败，可能参数不合法。")
        return 0

    print(f"\n[+] 曲线阶数 #E = {N}")

    # 2. 检查 Smart 攻击 (异常曲线)
    if N == p:
        print("[!] 警告: 曲线阶数恰好等于有限域特征 p (异常曲线)！")
        print("-> 威胁: 极易受到 Smart 攻击，可在毫秒级被破解。")
        return 0

    # 3. 提取最大素因子 n 和辅因子 h
    factors = factor(N)
    n = factors[-1][0]   # 取最大的素因子
    h = N // n
    print(f"[+] 最大素因子 n = {n} (比特长度: {n.nbits()})")
    print(f"[+] 辅因子 h = {h}")

    # 基准安全强度 (Pollard's rho)
    rho_security = n.nbits() / 2.0
    print(f"[*] 基于 Pollard's rho 算法的基准安全比特数: {rho_security:.1f} bits")

    # 4. MOV 攻击检测 (嵌入度 k)
    k_mov = None
    for k in range(1, 51):
        if (p**k - 1) % n == 0:
            k_mov = k
            break

    mov_security = rho_security
    if k_mov is not None:
        print(f"[!] 警告: 发现极小嵌入度 k = {k_mov}")
        # 扩域总比特数
        field_bits = (p**k_mov).bit_length()
        # 根据 NIST 标准估算扩域 DLP 安全强度
        if field_bits < 1024:
            mov_security = min(rho_security, 60)
        elif field_bits < 2048:
            mov_security = min(rho_security, 80)
        elif field_bits < 3072:
            mov_security = min(rho_security, 112)
        else:
            mov_security = min(rho_security, 128)
        print(f"-> 威胁: 易受 MOV 攻击，扩域总比特数为 {field_bits} bits。")
        print(f"[*] MOV 攻击降维后的安全比特数评估: ~{mov_security:.1f} bits")
    else:
        print("[+] 嵌入度 k > 50，对 MOV 攻击免疫。")

    # 5. 综合安全强度
    final_security = min(rho_security, mov_security)

    # 安全等级
    if final_security >= 128:
        level = "强安全"
    elif final_security >= 80:
        level = "弱安全"
    else:
        level = "不安全"

    print(f"\n{'='*45}")
    print(f"【结论】曲线 {curve_name} 的有效安全强度为: {final_security:.1f} bits")
    print(f"安全等级: {level}")
    print(f"{'='*45}\n")

    return final_security


# ========== 1. 验证 NIST P-256 曲线（强安全） ==========
# NIST P-256 参数 (FIPS 186-4)
p_p256 = 0xffffffff00000001000000000000000000000000ffffffffffffffffffffffff
a_p256 = -3
b_p256 = 0x5ac635d8aa3a93e7b3ebbd55769886bc651d06b0cc53b0f63bce3c3e27d2604b
# 也可使用 Sage 内置: E = EllipticCurve('P256')
evaluate_ecc_security(p_p256, a_p256, b_p256, curve_name="NIST P-256")

# ========== 2. 验证异常曲线（不安全） ==========
# 使用 PoliCTF 2012 公开的异常曲线参数 (p 约 160 位)
p_anom = 730750818665451459112596905638433048232067471723
a_anom = 425706413842211054102700238164133538302169176474
b_anom = 203362936548826936673264444982866339953265530166
evaluate_ecc_security(p_anom, a_anom, b_anom, curve_name="异常曲线 (PoliCTF 2012)")

# ========== 3. 验证随机弱曲线（弱安全，最大素因子约 160 位） ==========
# 生成一个 160 位素数 p，然后随机搜索一条曲线，使其最大素因子比特长度在 150~170 之间
print("\n--- 正在生成随机弱曲线（目标最大素因子约 160 位）---")
set_random_seed(42)  # 固定种子保证可重复性
found = False
for attempt in range(500):
    p_weak = random_prime(2^160, lbound=2^159)
    F = GF(p_weak)
    a = F.random_element()
    b = F.random_element()
    try:
        E = EllipticCurve(F, [a, b])
        N = E.order()
        if N == p_weak or E.is_supersingular():
            continue
        factors = factor(N)
        n_max = factors[-1][0]
        bits = n_max.nbits()
        if 150 <= bits <= 170:
            found = True
            break
    except:
        continue

if found:
    evaluate_ecc_security(p_weak, a, b, curve_name=f"随机弱曲线 (n 比特数={bits})")
else:
    print("未找到符合条件的弱曲线，请增加尝试次数或放宽范围。")


--- 正在评估曲线：NIST P-256 ---
p = 115792089210356248762697446949407573530086143415290314195533631308867097853951
a = -3
b = 41058363725152142129326129780047268409114441015993725554835256314039467401291

[+] 曲线阶数 #E = 115792089210356248762697446949407573529996955224135760342422259061068512044369
[+] 最大素因子 n = 115792089210356248762697446949407573529996955224135760342422259061068512044369 (比特长度: 256)
[+] 辅因子 h = 1
[*] 基于 Pollard's rho 算法的基准安全比特数: 128.0 bits
[+] 嵌入度 k > 50，对 MOV 攻击免疫。

【结论】曲线 NIST P-256 的有效安全强度为: 128.0 bits
安全等级: 强安全


--- 正在评估曲线：异常曲线 (PoliCTF 2012) ---
p = 730750818665451459112596905638433048232067471723
a = 425706413842211054102700238164133538302169176474
b = 203362936548826936673264444982866339953265530166

[+] 曲线阶数 #E = 730750818665451459112596905638433048232067471723
[!] 警告: 曲线阶数恰好等于有限域特征 p (异常曲线)！
-> 威胁: 极易受到 Smart 攻击，可在毫秒级被破解。

--- 正在生成随机弱曲线（目标最大素因子约 160 位）---

--- 正在评估曲线：随机弱曲线 (n 比特数=157) ---
p = 1105314955897002424102092374719285784905598362501
a = 107167857575222694